# HYDRA-Net Stage 1: Train on DroneRF Dataset

**Notebook purpose:** Train the XGBoost fast-triage classifier (Stage 1) on the real DroneRF dataset and produce publication-ready metrics.

**Runtime:** CPU is sufficient for Stage 1. No GPU required.

**Expected results (from literature):**
- 2-class (drone / no-drone): ~99% accuracy
- 4-class (drone type): ~84% accuracy
- 10-class (drone + flight mode): ~47% accuracy (this is the open problem)

**Dataset:** DroneRF from Al-Sa'd et al. (2019). [Paper link](https://www.sciencedirect.com/science/article/abs/pii/S0167739X18330760) · [Dataset on Mendeley Data](https://data.mendeley.com/datasets/f4c2b4n755/1)


## 1. Setup

In [ ]:
# Mount Google Drive to store the dataset (persists between sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the HYDRA-Net repo
!git clone https://github.com/YOUR-USERNAME/hydra-net.git
%cd hydra-net
!pip install -q -r requirements.txt

## 2. Download DroneRF dataset

The DroneRF dataset (~40 GB) contains RF signatures for 3 drone types (Bebop, AR, Phantom) across multiple flight modes. Download it once and cache to Drive.


In [ ]:
import os
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/hydra-net-data/dronerf')
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / 'RF_Data.csv').exists():
    # DroneRF is hosted on Mendeley Data — follow their direct download instructions.
    # See docs/DATASETS.md for the exact curl / wget commands.
    print('Please download DroneRF to', DATA_DIR)
    print('See docs/DATASETS.md for the download procedure.')
else:
    print('DroneRF already downloaded.')

## 3. Load DroneRF into HYDRA-Net feature format

DroneRF is distributed as `.csv` files with raw RF samples. We extract the same handcrafted feature vector that `hydra_net.stage1.features.extract_combined_features` produces.


In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from hydra_net.stage1 import FeatureConfig, extract_rf_features

# DroneRF sample rate (per Al-Sa'd et al.): 40 MHz
DRONERF_SAMPLE_RATE = 40e6
config = FeatureConfig(rf_sample_rate=DRONERF_SAMPLE_RATE)

# DroneRF file naming convention:
#   00000L.csv = background (no drone), lower band
#   10000L.csv / 10000H.csv = Bebop drone, mode 0
#   ... etc.
# See the DroneRF readme for the full mapping.

# Example loader (adapt to your downloaded file layout):
def load_dronerf_segment(csv_path, segment_length=1024):
    """Read a DroneRF CSV and chunk into segments for feature extraction."""
    raw = pd.read_csv(csv_path, header=None).values.flatten()
    n_segments = len(raw) // segment_length
    return [raw[i*segment_length:(i+1)*segment_length] for i in range(n_segments)]

# Iterate over all files, extract features, build X and y
# This cell is a template — fill in the file iteration for your specific layout.
print('Implement file iteration matching your DroneRF download layout.')

## 4. Train Stage 1

Once `X.npy` and `y.npy` are prepared, training is a one-liner using our `train_stage1.py` script:


In [ ]:
!PYTHONPATH=src python scripts/train_stage1.py \
    --X-path /content/drive/MyDrive/hydra-net-data/dronerf/X.npy \
    --y-path /content/drive/MyDrive/hydra-net-data/dronerf/y.npy \
    --model-output /content/drive/MyDrive/hydra-net-data/models/stage1_dronerf.json \
    --results-output results/stage1_dronerf.json \
    --confidence-threshold 0.95

## 5. Measure real exit rate and latency

This is the critical cascade metric — what fraction of DroneRF samples exit at Stage 1?


In [ ]:
!PYTHONPATH=src python scripts/benchmark_stage1_only.py \
    --stage1-model /content/drive/MyDrive/hydra-net-data/models/stage1_dronerf.json \
    --X-path /content/drive/MyDrive/hydra-net-data/dronerf/X.npy \
    --y-path /content/drive/MyDrive/hydra-net-data/dronerf/y.npy \
    --output results/stage1_dronerf_latency.json

## 6. Next steps

1. Open `02_stage2_antiuav_colab.ipynb` to train the cross-modal transformer on Anti-UAV (RGB + IR). Requires GPU.
2. Open `03_stage3_swarm_colab.ipynb` to train the swarm GNN. Requires GPU.
3. Open `04_end_to_end_benchmark.ipynb` to run the full cascade evaluation across all three stages.